# N127 · 质数筛、最小质因子与分解

**目标：** 按查询类型选择试除、筛或预处理。

**先修：** N126, N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 试除到平方根；Eratosthenes；线性筛导览；SPF；因子分解；质因数指数。

## 从问题到算法

逐数试除重复做很多整除检查。筛法从已知质数p开始标记其倍数；小于p²的倍数已有更小质因子处理，因此可从p²起步。最小质因子表不仅判断素性，还把多次分解查询变成不断除去最小因子的过程。

## 最小实现

**本章接口：** `sieve(n)`、`smallest_prime_factors(n)`、`factorize(x, spf)`

In [1]:
from math import isqrt

def sieve(n):
    if n<=2: return []
    prime=[True]*n; prime[0]=prime[1]=False
    for p in range(2,isqrt(n-1)+1):
        if prime[p]:
            for x in range(p*p,n,p): prime[x]=False
    return [x for x in range(2,n) if prime[x]]

def smallest_prime_factors(n):
    if n<0: raise ValueError('nonnegative bound required')
    spf=list(range(n+1))
    for p in range(2,isqrt(n)+1):
        if spf[p]==p:
            for x in range(p*p,n+1,p):
                if spf[x]==x: spf[x]=p
    return spf

def factorize(x,spf):
    if not 1<=x<len(spf): raise ValueError('x must be positive and covered by the SPF table')
    factors={}
    while x>1:
        p=spf[x]; factors[p]=factors.get(p,0)+1; x//=p
    return factors

## 正确性、前提与复杂度

每个合数都有不超过平方根的质因子，所以最终都会被标记；质数不会被更小质数整除。SPF只在尚未被标记时赋值，按p从小到大处理保证记录最小质因子。分解每轮严格减小x并保持“已取因子乘积×剩余x=原值”。

**代价：** 埃拉托斯特尼筛O(n log log n)算术工作量、O(n)空间；SPF同级预处理，分解最多O(log x)次整除。sieve返回严格小于n的质数，SPF表包含上界n，两个边界不混用。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

spf=smallest_prime_factors(30)
show_table(['整数','最小质因子','质因数分解'],[(x,spf[x],factorize(x,spf)) for x in range(2,21)])

整数,最小质因子,质因数分解
2,2,{2: 1}
3,3,{3: 1}
4,2,{2: 2}
5,5,{5: 1}
6,2,"{2: 1, 3: 1}"
7,7,{7: 1}
8,2,{2: 3}
9,3,{3: 2}
10,2,"{2: 1, 5: 1}"
11,11,{11: 1}


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert sieve(10)==[2,3,5,7] and sieve(2)==[] and sieve(0)==[]
spf=smallest_prime_factors(500)
def trial_prime(x): return x>=2 and all(x%d for d in range(2,isqrt(x)+1))
for x in range(1,501):
    factors=factorize(x,spf); product=1
    for p,e in factors.items(): assert trial_prime(p); product*=p**e
    assert product==x
    if x>=2: assert spf[x]==next(d for d in range(2,x+1) if x%d==0)
assert sieve(501)==[x for x in range(501) if trial_prime(x)]
print('N127: 所有本章断言通过')

N127: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释为什么从p²开始筛。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 批量分解与逐次试除比较。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 204. Count Primes（canonical）
- 2523. Closest Prime Numbers in Range（canonical）
- 2507. Smallest Value After Replacing With Sum of Prime Factors（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。